In [ ]:
from prompt_based_evaluator import PromptBasedEvaluator
import pandas as pd
from time import time
import asyncio

In [ ]:
df = pd.read_csv("exper_data.csv").iloc[0:25]

In [ ]:
models = [
    "OpenAI: GPT-5.4",
    "Anthropic: Claude Sonnet 4.6",
    "Google: Gemini 3 Flash Preview",
    "DeepSeek: DeepSeek V3.2",
    "Llama 4 Maverick",
    "Grok 4"
]

In [ ]:
rubric = """Correctness & Accuracy (25 points) — Ensures claims are factually accurate and verifiable, addressing the most critical concern of hallucination-free responses. This is weighted highest because inaccurate information undermines all other qualities.

Completeness (20 points) - Verifies the answer addresses all aspects of the query without significant omissions. This prevents shallow or partial responses that technically answer only part of the question.

Clarity & Coherence (18 points) - Assesses whether the answer is well-organized with logical flow. Research shows that coherence and relevance are strong signals of problem-solving quality.

Relevance (18 points) - Ensures all information pertains to the question, avoiding tangential content that confuses the issue. This maintains focus and efficiency.

Conciseness (10 points) - Rewards efficiency by penalizing unnecessary verbosity or repetition while maintaining completeness. This balances against verbose but complete responses.

Appropriateness for Context (9 points) — Checks whether tone, depth, and format match what the questioner likely needs. Technical questions require different treatment than conversational ones."""

In [ ]:
async def run_experiment_for(n):
    times = []
    results = []
    for _, row in df.iterrows():
        evaluator = PromptBasedEvaluator(*models[0:n])
        start = time()
        try:
            result = await evaluator.n_sq_evaluate(row["question"], rubric)
            end = time()
            times.append(end - start)
            results.append(result)
            print(f"Successfully evaluated question: {row['id']}")
        except Exception as e:
            print(f"Error evaluating question: {row['id']}")
            print(str(e))
    return results, sum(times) / len(times)

In [ ]:
results, n_2_time = await run_experiment_for(2)
print(f"Average time for n=2: {n_2_time:.2f} seconds")

In [ ]:
results, n_4_time = await run_experiment_for(4)
print(f"Average time for n=4: {n_4_time:.2f} seconds")

In [ ]:
results, n_6_time = await run_experiment_for(6)
print(f"Average time for n=6: {n_6_time:.2f} seconds")

# Times for each # of models
- 2 models = 2 22.38s
- 4 models = 56.14s
- 6 models = 4m 40.67s

As we can see here, when using the $n^2$ prompt based evaluation, which should theoretically be the worse of the 2 prompt based evaluators,
we get that our response time climbs steeply in relation to the number of models. A near 5 minute response time is unacceptable, so we will keep
the number of models capped at 4 for now.